# 03 — Earth observation GeoAI: Major TOM + NASA GIBS

**[🚀 Launch this notebook live](https://jltobias.github.io/JupyterLite-GeoLibre-GeoAI/lite/lab/index.html?path=v3_03_earth_observation_geoai.ipynb)**

This lab keeps the AI-ready **Major TOM Core** Sentinel-2, Copernicus DEM, and ESA WorldCover layers in GeoLibre, then runs a browser-safe unsupervised classification on a fixed **NASA GIBS MODIS Terra true-color** image.

> **Why the split?** Major TOM's Sentinel-2 chip is ZSTD-compressed. GeoLibre's browser raster engine can stream the COG, but the Rasterio/GDAL build currently available in Pyodide does not include the GDAL ZSTD GeoTIFF codec. Reading that TIFF with `rasterio.read()` raises `CPLE_AppDefinedError: ZSTD compression support is not configured`. The ML section therefore uses a JPEG response from NASA GIBS, avoiding GDAL compression codecs entirely.

> **JupyterLite compatibility:** maps use `geolibre_lite.LiteMap`, which avoids the localhost socket required by upstream `geolibre.Map`.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geolibre==3.0.0", "pyodide-http"])

from geolibre_lite import LiteMap as Map
assert Map.__module__ == "geolibre_lite", Map.__module__
print("Map implementation:", Map.__module__)


## Stream AI-ready Major TOM layers in GeoLibre

These COGs stay remote and are rendered by GeoLibre in the browser. We deliberately do **not** decode the ZSTD Sentinel-2 TIFF with Pyodide Rasterio.


In [ ]:
BASE = "https://data.source.coop/major-tom/core/DATA/42"
S2 = f"{BASE}/s2/data.tif"
DEM = f"{BASE}/dem/data.tif"
WC = f"{BASE}/wc/data.tif"

major_tom = Map(center=(8.0, 48.0), zoom=5, height="620px")
major_tom.add_cog(S2, name="Major TOM Sentinel-2 RGB", bands=[4, 3, 2], rescale=[0, 3000])
major_tom.add_cog(DEM, name="Major TOM Copernicus DEM", colormap="terrain")
major_tom.add_cog(WC, name="Major TOM ESA WorldCover")
major_tom


## Browser-safe EO classification from NASA GIBS

NASA GIBS exposes imagery through standards-based WMS/WMTS services. Here we request a fixed MODIS Terra true-color image over the San Francisco Bay region as JPEG, resize it to 128×128 pixels, and cluster RGB values with MiniBatch K-Means. The WMS bounding box gives us the georeferencing needed to map the cluster polygons.


In [ ]:
import io
import requests
import numpy as np
from PIL import Image
from IPython.display import display

if sys.platform == "emscripten":
    import pyodide_http
    pyodide_http.patch_all()

GIBS_WMS = "https://gibs.earthdata.nasa.gov/wms/epsg4326/best/wms.cgi"
BBOX = (-123.0, 37.0, -121.0, 39.0)  # west, south, east, north
DATE = "2024-07-15"
params = {
    "service": "WMS",
    "request": "GetMap",
    "version": "1.1.1",
    "layers": "MODIS_Terra_CorrectedReflectance_TrueColor",
    "styles": "",
    "format": "image/jpeg",
    "transparent": "false",
    "srs": "EPSG:4326",
    "bbox": ",".join(map(str, BBOX)),
    "width": 512,
    "height": 512,
    "time": DATE,
}

response = requests.get(GIBS_WMS, params=params, timeout=120)
response.raise_for_status()
source_image = Image.open(io.BytesIO(response.content)).convert("RGB")
print("GIBS image:", source_image.size, "date:", DATE)
display(source_image)


In [ ]:
from sklearn.cluster import MiniBatchKMeans

analysis_image = source_image.resize((128, 128), Image.Resampling.BILINEAR)
rgb = np.asarray(analysis_image, dtype=np.float32) / 255.0
pixels = rgb.reshape(-1, 3)

model = MiniBatchKMeans(
    n_clusters=6,
    random_state=42,
    n_init=10,
    batch_size=2048,
)
labels = model.fit_predict(pixels).reshape(rgb.shape[:2]).astype("int16")
np.unique(labels, return_counts=True)


In [ ]:
from rasterio.features import shapes
from rasterio.transform import from_bounds
import geopandas as gpd
from shapely.geometry import shape

h, w = labels.shape
transform = from_bounds(*BBOX, w, h)
records = [
    {"cluster": int(value), "geometry": shape(geom)}
    for geom, value in shapes(labels, transform=transform)
]

classes = gpd.GeoDataFrame(records, crs="EPSG:4326")
classes = classes.dissolve(by="cluster").reset_index()
classes["geometry"] = classes.geometry.simplify(0.01, preserve_topology=True)
classes[["cluster", "geometry"]]


In [ ]:
ml_map = Map(center=(-122.0, 38.0), zoom=7, height="650px")
ml_map.add_choropleth(
    classes,
    column="cluster",
    name=f"MODIS RGB K-Means — {DATE}",
    class_count=6,
    colormap="turbo",
    scheme="equal-interval",
    fillOpacity=0.55,
)
ml_map


## Interpretation

These are **RGB appearance clusters**, not semantic land-cover classes. Their purpose is to demonstrate a reproducible browser-native GeoAI pattern: retrieve EO imagery, convert pixels to features, fit an unsupervised model, georeference model output, and inspect the result in GeoLibre. A production workflow would use calibrated spectral bands, cloud masks, validation labels, and the full GeoAI/PyTorch stack where appropriate.

## Data & software citations

- Major TOM Core: https://source.coop/major-tom/core — AI-ready multimodal EO chips; example tile 42 supplies Sentinel-2, Copernicus DEM, and ESA WorldCover layers.
- Major TOM citation: Francis, A. & Czerkawski, M. (2024), *Major TOM: Expandable Datasets for Earth Observation*, IGARSS 2024, https://doi.org/10.1109/IGARSS53475.2024.10640760.
- NASA Global Imagery Browse Services (GIBS): https://nasa-gibs.github.io/gibs-api-docs/ — layer `MODIS_Terra_CorrectedReflectance_TrueColor`, fixed example date 2024-07-15.
- MODIS/Terra imagery is provided through NASA GIBS, part of NASA's Earth Science Data and Information System (ESDIS).
- Copernicus Sentinel-2 / Copernicus DEM and ESA WorldCover attribution follows the Major TOM source collection.
- GeoAI: Wu, Q. (2026), JOSS 11(118), 9605. https://doi.org/10.21105/joss.09605
- GeoLibre: https://geolibre.app/
